# Session 6: Loop Avoidance: STP, BPDU Guard and EtherChannel

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hamnakhan123/Cybersecurity-and-Digital-Forensics/blob/main/term-1/04-configuring-network-infrastructure/code/session-06-stp-and-etherchannel.ipynb)

*Cybersecurity and Digital Forensics* · Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run each cell from top to bottom (Shift+Enter). Every example is self-contained and safe to run in a lab or on Colab.

## `session-06_stp_election.py`

In [1]:
"""Session 6 - Spanning Tree Protocol: root bridge, root ports and blocked ports.

Bridge ID = priority (default 32768 + VLAN number) followed by the MAC address.
1. Lowest Bridge ID becomes ROOT; all its ports are designated (forwarding).
2. Every other switch picks one ROOT PORT: lowest cost to the root, then
   lowest sender Bridge ID, then lowest sender port ID.
3. On each segment, the switch with the lowest cost to root owns the
   DESIGNATED port. Every port that is neither root nor designated BLOCKS.
Port costs (short method): 10 Mbps 100, 100 Mbps 19, 1 Gbps 4, 10 Gbps 2.
Run:  python session-06_stp_election.py
"""
import heapq

COST = {10: 100, 100: 19, 1000: 4, 10000: 2}


def bid(sw):
    return (sw["priority"], sw["mac"])


def run_stp(switches, links):
    root = min(switches, key=lambda s: bid(switches[s]))
    # Dijkstra on path cost, ties broken by sender bridge ID then port name
    best = {root: (0, (), "")}          # switch -> (cost, sender_bid, local_port)
    pq = [(0, bid(switches[root]), root)]
    adj = {}
    for a, pa, b, pb, speed in links:
        adj.setdefault(a, []).append((b, pa, pb, COST[speed]))
        adj.setdefault(b, []).append((a, pb, pa, COST[speed]))
    done = set()
    while pq:
        c, _, u = heapq.heappop(pq)
        if u in done:
            continue
        done.add(u)
        for v, port_u, port_v, w in adj[u]:
            cand = (c + w, bid(switches[u]), port_v)
            if v not in best or cand < best[v]:
                best[v] = cand
                heapq.heappush(pq, (c + w, bid(switches[u]), v))
    roles = {}
    for a, pa, b, pb, speed in links:
        for sw, port in ((a, pa), (b, pb)):
            if sw != root and best[sw][2] == port:
                roles[(sw, port)] = "ROOT port       FWD"
        # designated port: side with lower root cost, then lower bridge ID
        side_a = (best[a][0], bid(switches[a])); side_b = (best[b][0], bid(switches[b]))
        des, dport, other, oport = (a, pa, b, pb) if side_a < side_b else (b, pb, a, pa)
        roles[(des, dport)] = "DESIGNATED port FWD"
        roles.setdefault((other, oport), "ALTERNATE port  BLOCKED")
    return root, best, roles


if __name__ == "__main__":
    switches = {"SW1": {"priority": 32769, "mac": "0011.1111.1111"},
                "SW2": {"priority": 32769, "mac": "0011.2222.2222"},
                "SW3": {"priority": 32769, "mac": "0011.0000.3333"}}
    links = [("SW1", "Gi0/1", "SW2", "Gi0/1", 1000),
             ("SW1", "Gi0/2", "SW3", "Gi0/1", 1000),
             ("SW2", "Gi0/2", "SW3", "Gi0/2", 1000)]
    for label in ("Default priorities (the oldest MAC wins - often a poor choice)",
                  "After 'spanning-tree vlan 1 root primary' on SW1 (priority 24577)"):
        root, best, roles = run_stp(switches, links)
        print(label)
        print(f"  Root bridge: {root}  (priority {switches[root]['priority']}, MAC {switches[root]['mac']})")
        for sw in sorted(switches):
            print(f"  {sw}: cost to root {best[sw][0]}")
            for (s, p), r in sorted(roles.items()):
                if s == sw:
                    print(f"     {p:6} {r}")
        print()
        switches["SW1"]["priority"] = 24577

Default priorities (the oldest MAC wins - often a poor choice)
  Root bridge: SW3  (priority 32769, MAC 0011.0000.3333)
  SW1: cost to root 4
     Gi0/1  DESIGNATED port FWD
     Gi0/2  ROOT port       FWD
  SW2: cost to root 4
     Gi0/1  ALTERNATE port  BLOCKED
     Gi0/2  ROOT port       FWD
  SW3: cost to root 0
     Gi0/1  DESIGNATED port FWD
     Gi0/2  DESIGNATED port FWD

After 'spanning-tree vlan 1 root primary' on SW1 (priority 24577)
  Root bridge: SW1  (priority 24577, MAC 0011.1111.1111)
  SW1: cost to root 0
     Gi0/1  DESIGNATED port FWD
     Gi0/2  DESIGNATED port FWD
  SW2: cost to root 4
     Gi0/1  ROOT port       FWD
     Gi0/2  ALTERNATE port  BLOCKED
  SW3: cost to root 4
     Gi0/1  ROOT port       FWD
     Gi0/2  DESIGNATED port FWD



## `session-06_etherchannel.py`

In [2]:
"""Session 6 - EtherChannel: which modes form a bundle, and how traffic is shared.

PAgP (Cisco):  desirable + desirable/auto form a channel; auto + auto does not.
LACP (802.3ad): active + active/passive form a channel; passive + passive does not.
'on' forms only with 'on' (no negotiation at all).
Traffic is NOT split packet by packet: a hash of addresses chooses one member
link per flow, so a single flow never goes faster than one link.
Run:  python session-06_etherchannel.py
"""
import hashlib
from collections import Counter

FORMS = {
    ("desirable", "desirable"): "PAgP channel", ("desirable", "auto"): "PAgP channel",
    ("auto", "auto"): "no channel (both wait)",
    ("active", "active"): "LACP channel", ("active", "passive"): "LACP channel",
    ("passive", "passive"): "no channel (both wait)",
    ("on", "on"): "static channel",
}


def negotiate(a, b):
    return FORMS.get((a, b)) or FORMS.get((b, a)) or "no channel (modes do not match)"


def member_for(src_ip, dst_ip, links=4):
    """Simplified src-dst-ip hash: same pair of addresses -> same physical link."""
    h = hashlib.sha256(f"{src_ip}-{dst_ip}".encode()).digest()[0]
    return h % links


if __name__ == "__main__":
    print("Mode compatibility")
    for a, b in [("desirable", "auto"), ("auto", "auto"), ("active", "passive"),
                 ("passive", "passive"), ("on", "active"), ("on", "on")]:
        print(f"  {a:9} + {b:9} -> {negotiate(a, b)}")

    print("\nLoad sharing over a 4 x 1 Gbps Port-channel (src-dst-ip hash)")
    flows = [(f"10.1.10.{h}", "10.1.99.10") for h in range(1, 41)]
    use = Counter(member_for(s, d) for s, d in flows)
    for link in range(4):
        print(f"  Gi1/0/{link+1}: {use[link]:2} flows  " + "#" * use[link])
    print("  One backup flow 10.1.10.7 -> 10.1.99.10 always uses Gi1/0/"
          f"{member_for('10.1.10.7', '10.1.99.10') + 1} (max 1 Gbps for that flow)")
    print("\nBundled bandwidth seen by STP: 4 Gbps as ONE logical port, so no port blocks.")

Mode compatibility
  desirable + auto      -> PAgP channel
  auto      + auto      -> no channel (both wait)
  active    + passive   -> LACP channel
  passive   + passive   -> no channel (both wait)
  on        + active    -> no channel (modes do not match)
  on        + on        -> static channel

Load sharing over a 4 x 1 Gbps Port-channel (src-dst-ip hash)
  Gi1/0/1:  9 flows  #########
  Gi1/0/2: 11 flows  ###########
  Gi1/0/3:  7 flows  #######
  Gi1/0/4: 13 flows  #############
  One backup flow 10.1.10.7 -> 10.1.99.10 always uses Gi1/0/3 (max 1 Gbps for that flow)

Bundled bandwidth seen by STP: 4 Gbps as ONE logical port, so no port blocks.
